# 视频配音流水线：英文演讲 / 访谈 → 中文配音

把英文 TED 演讲、访谈等视频转成**中文配音版**（IndexTTS-2.5 克隆原声音色）。

**会做什么**
- 用人声分离保住掌声、背景音乐
- Whisper 转写 + 说话人分离（可多人）
- 有中/英字幕则优先用字幕当稿，并去掉译者、说话人标签等不该念的字
- 有清洗后的字幕就跳过 Whisper；说话人优先用字幕标签，否则单人/分离
- DeepSeek 把英文译成适合配音的中文
- 按说话人克隆音色，中文时长尽量对齐画面
- 中途断开可从检查点接着跑；某一句读错可只重做那一句

**开始前请准备**
1. **Runtime → 更改运行时类型 → GPU（T4 即可）**
2. 在 [Colab 密钥](https://colab.research.google.com/notebooks/secrets.ipynb) 里添加：
   - `LLM_API_KEY`：DeepSeek API Key（[platform.deepseek.com](https://platform.deepseek.com)）
   - `HF_TOKEN`：HuggingFace Token。仅当说话人 ≥ 2 时需要；并同意
     [pyannote 分离模型](https://huggingface.co/pyannote/speaker-diarization-3.1) 与
     [分割模型](https://huggingface.co/pyannote/segmentation-3.0) 的使用条款
3. 单人讲座可把下面的 `NUM_SPEAKERS` 设为 `1`，不必填 `HF_TOKEN`

## 0. 挂载 Google 云端硬盘（模型缓存）

大模型（IndexTTS-2.5、Whisper、Demucs、Pyannote）会写到
`我的云端硬盘/index-tts-cache/`，下次开 session 不用重下。
请点「连接到 Google 云端硬盘」并允许访问。

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Model cache on Google Drive (avoids re-downloading every session) ---
DRIVE_CACHE = "/content/drive/MyDrive/index-tts-cache"
os.makedirs(f"{DRIVE_CACHE}/hf_home", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/torch_home", exist_ok=True)

# HuggingFace models: Whisper, Pyannote, WhisperX alignment, IndexTTS2
# Only set HF_HOME — HF_HUB_CACHE auto-derives as HF_HOME/hub
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
# PyTorch hub models: Demucs
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

print(f"Model cache: {DRIVE_CACHE}")
print(f"HF_HOME:    {os.environ['HF_HOME']}")
print(f"TORCH_HOME: {os.environ['TORCH_HOME']}")

## 1. 检查 GPU、克隆仓库、安装依赖

安装单元格跑完会**自动重启运行时**。重启后不要重装，只跑下一格「恢复环境」。

In [ ]:
!nvidia-smi
import sys
print(f"Python {sys.version}")

In [ ]:
# 克隆 py3.12 分支（已有仓库则拉取更新）
%cd /content
!git clone -b py3.12 https://github.com/deluxebear/index-tts.git 2>/dev/null || (cd /content/index-tts && git pull)
%cd /content/index-tts

In [ ]:
%cd /content/index-tts
!bash tools/setup_colab.sh --extra dub

print("Restarting runtime to apply package changes...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

### 运行时重启后：先跑这一格恢复环境

上一格会关掉内核。从这里继续：重新挂载硬盘、回到项目目录、确认 torch/cuda 正常。**不要再跑安装格。**

In [ ]:
import os
from google.colab import drive

# Re-mount Drive and restore env vars after runtime restart
drive.mount('/content/drive')

DRIVE_CACHE = "/content/drive/MyDrive/index-tts-cache"
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

%cd /content/index-tts

# Verify packages loaded correctly
import torch, numpy, numba
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")
print(f"numpy={numpy.__version__}")
print(f"numba={numba.__version__}")
print("All good!")

## 2. 下载 IndexTTS-2.5 权重

权重大约数 GB，下到云端硬盘的 `index-tts-cache/checkpoints-2.5/`，再软链到项目里的 `checkpoints/`。
已经下过会自动跳过。

In [ ]:
import os

DRIVE_CKPT = f"{DRIVE_CACHE}/checkpoints-2.5"
os.environ["INDEX_TTS_MODEL_DIR"] = DRIVE_CKPT
LOCAL_CKPT = "/content/index-tts/checkpoints"

# Download to Drive (persists across sessions)
if not os.path.exists(f"{DRIVE_CKPT}/config.yaml"):
    print("Downloading IndexTTS2 checkpoints to Google Drive (first time only)...")
    !huggingface-cli download IndexTeam/IndexTTS-2.5 --local-dir "{DRIVE_CKPT}"
else:
    print(f"Checkpoints already cached at {DRIVE_CKPT}")

# Symlink so the pipeline finds them at the expected local path
if os.path.islink(LOCAL_CKPT):
    os.unlink(LOCAL_CKPT)
elif os.path.exists(LOCAL_CKPT):
    import shutil
    shutil.rmtree(LOCAL_CKPT) if os.path.isdir(LOCAL_CKPT) else os.remove(LOCAL_CKPT)
os.symlink(DRIVE_CKPT, LOCAL_CKPT)
print(f"Symlinked: {LOCAL_CKPT} -> {DRIVE_CKPT}")

# Verify
!ls -la checkpoints/config.yaml

## 3. 配置密钥与配音参数

`LLM_API_KEY` 默认走 **DeepSeek**（`deepseek-v4-flash`）。密钥在 Colab「密钥」里，名称必须是 `LLM_API_KEY`。

| 参数 | 含义 |
|---|---|
| `NUM_SPEAKERS` | 说话人数。有字幕时跳过 Whisper；`1` 不跑 pyannote |
| `AUDIO_ONLY_ALIGN` | `True`：中文偏长只加速音频，不重编码画面（更快） |
| `WHISPER_MODEL` | 转写模型。`large-v3-turbo` 更快；要精度可改 `large-v2` |
| `EXTERNAL_SUBS` | 指定字幕路径；也可把 `视频名.srt` 放在视频同目录 |
| `WORK_DIR` | 中间文件目录。默认写云盘，断线可续跑 |
| `CLEANUP` | `False` 才能事后改某一句。不要开 `True` 除非确定不再修补 |

In [ ]:
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')     # 仅无说话人标签的多人分离需要
except Exception:
    HF_TOKEN = None
try:
    LLM_API_KEY = userdata.get('LLM_API_KEY')  # 中文字幕可省略
except Exception:
    LLM_API_KEY = None

# --- 翻译模型（默认 DeepSeek；要换就注释/取消注释） ---
LLM_API_BASE = "https://api.deepseek.com/v1"    ; LLM_MODEL = "deepseek-v4-flash"
# LLM_API_BASE = "https://api.openai.com/v1"       ; LLM_MODEL = "gpt-4o-mini"
# LLM_API_BASE = "https://generativelanguage.googleapis.com/v1beta/openai/" ; LLM_MODEL = "gemini-2.0-flash"

# --- 配音参数 ---
NUM_SPEAKERS = 2           # 1=不跑 pyannote；有字幕时任何人声数都跳过 Whisper
USE_FP16 = True
WORK_DIR = f"{DRIVE_CACHE}/dub_workspace"  # 默认写云盘，断线可续跑
CLEANUP = False            # 必须 False，才能事后只改某一句
AUDIO_ONLY_ALIGN = True    # True=不重编码画面；False=必要时放慢画面
WHISPER_MODEL = "large-v3-turbo"  # 要更准可改 "large-v2"
EXTERNAL_SUBS = None       # 例如 "/content/talk.zh.srt"；或把字幕放视频旁边

print(f"LLM: {LLM_MODEL} @ {LLM_API_BASE}")
print(f"Speakers={NUM_SPEAKERS} whisper={WHISPER_MODEL} audio_only_align={AUDIO_ONLY_ALIGN}")
print(f"WORK_DIR={WORK_DIR}")

## 4. 预下载大模型（可选）

第一次建议跑：把 Whisper、Demucs、Pyannote 拉到云端硬盘。之后 session 会直接用缓存。
单人（`NUM_SPEAKERS=1`）不会去下 Pyannote。

In [ ]:
# Pre-download Whisper / Demucs / Pyannote to Drive (optional)
import whisperx
print(f"Loading Whisper {WHISPER_MODEL} (cached on Drive)...")
_model = whisperx.load_model(WHISPER_MODEL, "cuda", compute_type="float16")
del _model
print("Whisper model cached.")

import demucs.pretrained
print("Loading Demucs model (cached on Drive)...")
_model = demucs.pretrained.get_model("htdemucs")
del _model
print("Demucs model cached.")

if NUM_SPEAKERS != 1:
    from whisperx.diarize import DiarizationPipeline
    print("Loading Pyannote model (cached on Drive)...")
    _diarize = DiarizationPipeline(token=HF_TOKEN, device="cuda")
    del _diarize
    print("Pyannote model cached.")

import torch; torch.cuda.empty_cache()
print("\nAll models cached on Google Drive. Future sessions will start faster.")

## 5. 处理一条视频

已有 `video_path` 且文件还在时直接续跑，不会再弹上传。换片把 `FORCE_UPLOAD = True`，或改用下面的 Drive 路径。

**字幕（推荐）**：把 `视频名.srt` / `.ass` 和视频放一起，或设置 `EXTERNAL_SUBS`。
- 中文字幕：清洗后直接当配音稿（去掉译者、说话人标签、掌声等）
- 英文字幕：清洗后由 DeepSeek 翻译，时间轴用字幕的
- 有清洗后的字幕：跳过 Whisper。字幕里的 `张三：` / `CA:` 用来分说话人；没有标签时单人全归一人，多人只跑 pyannote
- 没有字幕：Whisper 听写再翻译

同一条视频中途失败，用相同 `video_path` 和 `WORK_DIR` 再跑，会从检查点接着做。

In [ ]:
from dub_pipeline import dub_video
from google.colab import files
from pathlib import Path
import os

# 续跑：保留已有 video_path。换片：FORCE_UPLOAD = True，或改用下面的 Drive 路径。
FORCE_UPLOAD = False
# video_path = "/content/drive/MyDrive/videos/ted_talk.mp4"

if FORCE_UPLOAD or "video_path" not in dir() or not Path(str(video_path)).is_file():
    uploaded = files.upload()
    video_path = list(uploaded.keys())[0]

# === 开始配音（中断后再跑会自动续）===
stem = Path(video_path).stem
os.makedirs(f"{DRIVE_CACHE}/output", exist_ok=True)
output_path = f"{DRIVE_CACHE}/output/{stem}_cn.mp4"

dub_video(
    video_path=video_path,
    output_path=output_path,
    work_dir=WORK_DIR,
    hf_token=HF_TOKEN,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    model_dir="checkpoints",
    num_speakers=NUM_SPEAKERS,
    use_fp16=USE_FP16,
    cleanup=CLEANUP,
    audio_only_align=AUDIO_ONLY_ALIGN,
    whisper_model=WHISPER_MODEL,
    external_subs=EXTERNAL_SUBS,
)

# Play result inline
from IPython.display import Video, display
display(Video(output_path, embed=True, width=640))

In [ ]:
# 下载配好的中文视频到本地
files.download(output_path)

## 6. 批量处理云盘文件夹

把待配视频放进 `INPUT_DIR`，成品写到 `OUTPUT_DIR`。每个视频单独工作目录，互不影响；已有 `_cn` 后缀的会跳过。
默认人数用上面的 `NUM_SPEAKERS`。个别片子人数不同时，填 `NUM_SPEAKERS_FOR`：文件名、不带后缀的名字、或相对路径都可以。

In [ ]:
from dub_pipeline import dub_batch

INPUT_DIR  = "/content/drive/MyDrive/videos/input"   # 待配视频
OUTPUT_DIR = "/content/drive/MyDrive/videos/output"  # 中文成品

# 个别视频覆盖人数（没写的用上面的 NUM_SPEAKERS）
NUM_SPEAKERS_FOR = {
    # "ted_talk.mp4": 1,
    # "panel.mp4": 3,
    # "interviews/qna.mp4": 4,
}

dub_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    hf_token=HF_TOKEN,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    work_dir=WORK_DIR,
    model_dir="checkpoints",
    num_speakers=NUM_SPEAKERS,
    num_speakers_map=NUM_SPEAKERS_FOR,
    use_fp16=USE_FP16,
    cleanup=CLEANUP,
    audio_only_align=AUDIO_ONLY_ALIGN,
    whisper_model=WHISPER_MODEL,
    external_subs=EXTERNAL_SUBS,
)
print(f"\n全部完成。成品目录: {OUTPUT_DIR}")

---
## 7. 只改某一句（重新 TTS + 封装）

整片跑完后，若某句多音字或专名读错，不必重跑流水线。
先列出句子编号，再只重做那几句。可用 IndexTTS-2.5 发音标注：

- 拼音（须在 `checkpoints/pinyin.vocab`）：`他在银<行|HANG2>办理业务`
- 英文 CMU 音素：`我们用<ChatGPT|CH AE1 T JH IY1 P IY1 T IY1>`

需要 `WORK_DIR` 里还有该视频的 `checkpoint.json`（所以 `CLEANUP` 必须是 `False`）。

In [ ]:
from dub_pipeline import list_dub_segments, list_suspicious_segments

# 使用上一格的 video_path
list_dub_segments(video_path, work_dir=WORK_DIR)
print("\n--- 可能需要重配的句子 ---")
list_suspicious_segments(video_path, work_dir=WORK_DIR)

In [ ]:
from dub_pipeline import redub_segments

REDUB_IDS = [12]  # 改成 list 格里看到的编号
REDUB_TEXT = "他在银<行|HANG2>办理业务"  # 改成 None 则沿用已有中文

redub_segments(
    video_path,
    REDUB_IDS,
    texts={REDUB_IDS[0]: REDUB_TEXT} if REDUB_TEXT else None,
    output_path=output_path,
    work_dir=WORK_DIR,
    model_dir="checkpoints",
    use_fp16=USE_FP16,
)
from IPython.display import Video, display
display(Video(output_path, embed=True, width=640))

## 8. 查看中间结果

中间文件在 `WORK_DIR/视频名/`（默认云盘 `index-tts-cache/dub_workspace/`）：转写、译文、人声、各句 wav 等。
专名读音可改仓库根目录或 `WORK_DIR` 下的 `pronunciation.yaml`。

In [ ]:
import json, os
from IPython.display import Audio, display

# 打开最近一次配音的工作目录
work_dirs = sorted(
    [d for d in os.listdir(WORK_DIR) if os.path.isdir(f"{WORK_DIR}/{d}")]
) if os.path.isdir(WORK_DIR) else []

if work_dirs:
    work_dir = f"{WORK_DIR}/{work_dirs[-1]}"
    print(f"Work directory: {work_dir}")
    print(f"Contents: {os.listdir(work_dir)}")

    # Show transcript
    transcript_path = f"{work_dir}/transcript.json"
    if os.path.exists(transcript_path):
        with open(transcript_path) as f:
            segments = json.load(f)
        print(f"\n--- 转写 ({len(segments)} 段) ---")
        for s in segments[:10]:
            print(f"  [{s.get('speaker','?')}] {s['start']:.1f}-{s['end']:.1f}s: {s['text']}")
        if len(segments) > 10:
            print(f"  ... and {len(segments)-10} more")

    # Show translations
    trans_path = f"{work_dir}/translations.json"
    if os.path.exists(trans_path):
        with open(trans_path) as f:
            trans = json.load(f)
        print(f"\n--- 译文 ---")
        for t in trans[:10]:
            print(f"  #{t['id']} [{t.get('speaker','?')}] EN: {t['en'][:50]}")
            print(f"       ZH: {t['zh'][:50]}")

    # Play separated vocals
    vocals_dir = f"{work_dir}/htdemucs"
    if os.path.isdir(vocals_dir):
        for subdir in os.listdir(vocals_dir):
            voc = f"{vocals_dir}/{subdir}/vocals.wav"
            if os.path.exists(voc):
                print(f"\n--- 分离出的人声 ---")
                display(Audio(voc))
                break
else:
    print("还没有工作目录。请先跑第 5 节配一条视频。")